# Logits Preprocessing and Data Engineering

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/extraction',
            'transformation': 'curated',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'output_path' : '/workspaces/CodeSmells/datax/code_smells/logits',
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [ ]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [3]:
from transformers import CodeLlamaTokenizer, LlamaForCausalLM
from datasets import load_dataset

In [ ]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['transformation']}"
create_folder(log_file)
log_file += '/data_en.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [ ]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Dataset

In [ ]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['transformation']}_{params['dataset']['sampling_size']}.json", )

In [ ]:
df_dataset = df_dataset.reset_index(drop=True, inplace=True)

#### Model Loading

In [9]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = CodeLlamaTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [10]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### preprocess dataset

In [ ]:
df_dataset['input_ids'] = df_dataset[params['content_column']].map(lambda code: tokenizer.encode(code))
df_dataset['len_input_ids'] = df_dataset[params['input_ids']].map(lambda input_ids: len(input_ids))

#### Softmax Normalization and Data Engineering

In [11]:
def topk_tuple( logit_vocab_tensor, largest, tokenizer_fn):
    "Run topk for a token"
    topk = logit_vocab_tensor.topk( k=1 , largest=largest ) #TODO K number of elements can be extended
    return ( tokenizer.convert_tokens_to_string([tokenizer_fn.decode(topk.indices)]), topk.values.item())

def min_max_logits( logit_vocab_sample_tensor, tokenizer_fn ):
    "Compute min_max for a sample"
    max_cases = []
    min_cases = []
    for logit_vocab_tensor in logit_vocab_sample_tensor:
        max_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = True, tokenizer_fn = tokenizer_fn) ) #TST Max Logit
        min_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = False, tokenizer_fn = tokenizer_fn) ) #TST Min Logit
    return max_cases, min_cases

def actual_logit( 
                 logit_vocab_sample_tensor, 
                 tokenized_prompt, 
                 tokenizer_fn,
                 ):
    "Compute actual logits for a sample"
    actual_logits_prompt = []
    for token_pos, id_token in enumerate( tokenized_prompt[1:] ): #Eliminate the first token prediction since we do not use it
        actual_logits_prompt.append(
            (   tokenizer.convert_tokens_to_string([tokenizer_fn.decode( int(id_token))]), #retrieving the name of the token with the id
                logit_vocab_sample_tensor[token_pos][int(id_token)].item()) #retrieving the logit given the position in the sequence and the position in the vocab
            )
    return actual_logits_prompt

In [12]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [ ]:
callbacks_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
out = np.load(f"{callbacks_dir}/logits_tensor[0]_batch[0].npy")

print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 224, 32016)


In [14]:
max_case,min_case = min_max_logits(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out], ####### 
    tokenizer_fn= tokenizer
    )
print(max_case)
assert len(max_case) == len(min_case)

[('<PRE>', 0.7492888569831848), ('module', 0.4794430136680603), ('_', 0.8005598783493042), ('get', 0.03342306241393089), ('_', 0.7154065370559692), ('of', 0.029203299432992935), ('(', 0.4235145151615143), ('to', 0.047612469643354416), ('bin', 0.33591780066490173), ('ations', 0.5140894055366516), ('(', 0.5055526494979858), ('self', 0.23513413965702057), ('):', 0.8747055530548096), ('\n', 0.9750158190727234), ('      ', 0.63499915599823), ('"""', 0.23746822774410248), ('=', 0.9357321858406067), ('[', 0.34527602791786194), ('1', 0.581474244594574), (',', 0.7503364682197571), ('', 0.7880880832672119), ('2', 0.6225070953369141), (',', 0.8709840774536133), ('', 0.9746920466423035), ('3', 0.9141796231269836), (']', 0.5397116541862488), ('', 0.9849329590797424), ('4', 0.9219658970832825), (']', 0.7163926959037781), ('\n', 0.990165650844574), ('      ', 0.953723669052124), ('assert', 0.1824035793542862), ('=', 0.9945827126502991), ('[', 0.8366445302963257), ('1', 0.24499067664146423), (',', 0.9

In [ ]:
assert tokenizer.decode(df_dataset['input_ids'][0]) == df_dataset[params['dataset']['content_column']][0]
df_dataset[params['dataset']['content_column']][0]

2024-09-30 12:58:21.798868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-30 12:58:21.813189: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-30 12:58:21.817604: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-09-30 12:58:21.828777: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


'def test_array_sum_combinations(self):\n        A = [1, 2, 3, 3]\n        B = [2, 3, 3, 4]\n        C = [2, 3, 3, 4]\n        target = 7\n        answer = [[1, 2, 4], [1, 3, 3], [1, 3, 3], [1, 3, 3],\n                  [1, 3, 3], [1, 4, 2], [2, 2, 3], [2, 2, 3],\n                  [2, 3, 2], [2, 3, 2], [3, 2, 2], [3, 2, 2]]\n        answer.sort()\n        self.assertListEqual(sorted(array_sum_combinations(A, B, C, target)),\n                             answer)\n'

In [16]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

actual_cases = actual_logit(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    tokenized_prompt = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
    )
actual_cases

[('def', 0.000761137343943119),
 ('test', 0.019638611003756523),
 ('_', 0.8005598783493042),
 ('array', 0.0013487939722836018),
 ('_', 0.7154065370559692),
 ('sum', 0.010651934891939163),
 ('_', 0.25596386194229126),
 ('com', 0.003988292068243027),
 ('bin', 0.33591780066490173),
 ('ations', 0.5140894055366516),
 ('(', 0.5055526494979858),
 ('self', 0.23513413965702057),
 ('):', 0.8747055530548096),
 ('\n', 0.9750158190727234),
 ('      ', 0.63499915599823),
 ('A', 0.005719942040741444),
 ('=', 0.9357321858406067),
 ('[', 0.34527602791786194),
 ('1', 0.581474244594574),
 (',', 0.7503364682197571),
 ('', 0.7880880832672119),
 ('2', 0.6225070953369141),
 (',', 0.8709840774536133),
 ('', 0.9746920466423035),
 ('3', 0.9141796231269836),
 (',', 0.45317918062210083),
 ('', 0.9849329590797424),
 ('3', 0.006993327289819717),
 (']', 0.7163926959037781),
 ('\n', 0.990165650844574),
 ('      ', 0.953723669052124),
 ('B', 0.1225515678524971),
 ('=', 0.9945827126502991),
 ('[', 0.8366445302963257),


#### Processing all the Batches

In [ ]:
def batching_logits(tokenizer,tf_input_ids,size=10000):
    max_logit_token_prompt = []
    min_logit_token_prompt = []
    actual_logit_token_prompt = []

    
    soft = torch.nn.Softmax( dim = 0 )                          #Flattening normalization
    
    for file in range( size ):
        callbacks_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
        out = np.load(f"{callbacks_dir}/logits_tensor[{file}]_batch[{file}].npy") #<sample,tokens,voc_tokens>
        out = out[0]  ##### #<tokens,voc_tokens>
        next_tokens_distribution = [ soft( torch.from_numpy(token) ) for token in out]  #Flattening normalization
        
        max_cases,min_cases = min_max_logits(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenizer_fn= tokenizer
            )

        actual_cases = actual_logit(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenized_prompt = tf_input_ids[ file ],
            tokenizer_fn = tokenizer
            )
        
        max_logit_token_prompt.append( max_cases )
        min_logit_token_prompt.append( min_cases )
        actual_logit_token_prompt.append( actual_cases )
        
        logging.info(file)
    return max_logit_token_prompt,min_logit_token_prompt,actual_logit_token_prompt

In [18]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [19]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = batching_logits(
    tokenizer=tokenizer , tf_input_ids=input_ids_list, 
    size = len(df_dataset)
) #<---WARNING TIME Consuming

IndexError: list index out of range

#### Saving results

In [18]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [19]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(55, 30)

In [20]:
dataframe_to_save.head(5)

,msg_id,line,column,end_line,end_column,code_smell,code,func_name,commit_id,repo,...,vocab_size,nloc,token_counts,n_identifiers,repository,year,input_ids,max_prob,min_prob,actual_prob
0,C0103,1,0,1,12,def test_DFA(self):,def test_DFA(self):\n transitions = {\n...,test_DFA,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,64,29,264,16,algorithms,outputs-22,"[822, 1243, 29918, 4037, 29909, 29898, 1311, 1...","[(<PRE>, 0.7492899298667908), (module, 0.47944...","[(<s>, 6.321468413832132e-13), ($}, 1.21012033...","[(def, 0.0007611322216689587), (test, 0.019638..."
1,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,28,11,165,11,algorithms,outputs-22,"[822, 1243, 29918, 2378, 29918, 2083, 29918, 5...","[(<PRE>, 0.7492888569831848), (module, 0.47944...","[(<s>, 6.321785000866498e-13), ($}, 1.21012561...","[(def, 0.000761137343943119), (test, 0.0196386..."
2,C0103,3,8,3,9,"B = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,28,11,165,11,algorithms,outputs-22,"[822, 1243, 29918, 2378, 29918, 2083, 29918, 5...","[(<PRE>, 0.7492888569831848), (module, 0.47944...","[(<s>, 6.321785000866498e-13), ($}, 1.21012561...","[(def, 0.000761137343943119), (test, 0.0196386..."
3,C0103,4,8,4,9,"C = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,28,11,165,11,algorithms,outputs-22,"[822, 1243, 29918, 2378, 29918, 2083, 29918, 5...","[(<PRE>, 0.7492888569831848), (module, 0.47944...","[(<s>, 6.321785000866498e-13), ($}, 1.21012561...","[(def, 0.000761137343943119), (test, 0.0196386..."
4,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_unique_array_sum_combinations(self):\...,test_unique_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,30,11,117,11,algorithms,outputs-22,"[822, 1243, 29918, 13092, 29918, 2378, 29918, ...","[(<PRE>, 0.749289333820343), (module, 0.479442...","[(<s>, 6.322174771547506e-13), ($}, 1.21012075...","[(def, 0.0007611389737576246), (test, 0.019638..."


In [ ]:
output_dir = f"{params['output_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
create_folder(output_dir)
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

#### Loss Retrieval

In [ ]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_dir = f"{params['output_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
    output_loss = []
    for current_batch in range(size):
        out = np.load(f"{output_dir}/_loss_batch[{current_batch}].npy")
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [23]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [24]:
output_loss

[0.4024338722229004,
 0.571848452091217,
 0.571848452091217,
 0.571848452091217,
 0.7669380307197571,
 0.7669380307197571,
 0.7669380307197571,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.6598173379898071,
 0.7339152693748474,
 0.20522019267082214,
 1.5347800254821777,
 1.3996508121490479,
 1.4757285118103027,
 0.720551609992981,
 0.994199812412262,
 1.055303692817688,
 0.5430501699447632,
 0.5430501699447632,
 0.5430501699447632,
 0.5430501699447632,
 0.5430501699447632,
 0.5430501699447632,
 1.188644289970398,
 1.0981333255767822,
 1.589848518371582,
 1.826017141342163,
 1.826017141342163,
 1.5603570938110352,
 1.5603570938110352,
 2.572435140609741,
 0.8118057250976562,
 0.8118057250976562,
 0.2825244069099426,
 0.2825244069099426,
 0.9048572778701782,
 0.6807515621185303,
 0.740659236907959,
 1.1595052480697632,
 1.2384271621704102,
 1.

In [25]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,msg_id,line,column,end_line,end_column,code_smell,code,func_name,commit_id,repo,...,nloc,token_counts,n_identifiers,repository,year,input_ids,max_prob,min_prob,actual_prob,loss
0,C0103,1,0,1,12,def test_DFA(self):,def test_DFA(self):\n transitions = {\n...,test_DFA,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,29,264,16,algorithms,outputs-22,"[822, 1243, 29918, 4037, 29909, 29898, 1311, 1...","[(<PRE>, 0.7492899298667908), (module, 0.47944...","[(<s>, 6.321468413832132e-13), ($}, 1.21012033...","[(def, 0.0007611322216689587), (test, 0.019638...",0.402434
1,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[822, 1243, 29918, 2378, 29918, 2083, 29918, 5...","[(<PRE>, 0.7492888569831848), (module, 0.47944...","[(<s>, 6.321785000866498e-13), ($}, 1.21012561...","[(def, 0.000761137343943119), (test, 0.0196386...",0.571848
2,C0103,3,8,3,9,"B = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[822, 1243, 29918, 2378, 29918, 2083, 29918, 5...","[(<PRE>, 0.7492888569831848), (module, 0.47944...","[(<s>, 6.321785000866498e-13), ($}, 1.21012561...","[(def, 0.000761137343943119), (test, 0.0196386...",0.571848
3,C0103,4,8,4,9,"C = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[822, 1243, 29918, 2378, 29918, 2083, 29918, 5...","[(<PRE>, 0.7492888569831848), (module, 0.47944...","[(<s>, 6.321785000866498e-13), ($}, 1.21012561...","[(def, 0.000761137343943119), (test, 0.0196386...",0.571848
4,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_unique_array_sum_combinations(self):\...,test_unique_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,117,11,algorithms,outputs-22,"[822, 1243, 29918, 13092, 29918, 2378, 29918, ...","[(<PRE>, 0.749289333820343), (module, 0.479442...","[(<s>, 6.322174771547506e-13), ($}, 1.21012075...","[(def, 0.0007611389737576246), (test, 0.019638...",0.766938


In [ ]:
## Saving CheckPoint 2
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

In [27]:
torch.cuda.empty_cache()
gc.collect()

37